# MVCBM — diagnosis apendisitis pediatrik (training di Kaggle)

Notebook ini hanya memanggil fungsi dari file `.py` di repo `pediatric-kumadai`. Kode diedit secara lokal, di-push ke GitHub, lalu dijalankan di sini.

Sebelum *Run All*:
1. **Settings → Accelerator → GPU** (T4).
2. **Settings → Internet → On** (untuk `git clone`).
3. **Add Input** → dataset Kaggle (Private) berisi folder `preprocessed_400/` dan `dicts_diagnosis/imputed/final/` (`app_data_train`, `app_data_test`) hasil `prepare_data.py`.
4. Jika repo GitHub private: **Add-ons → Secrets** → tambahkan `GITHUB_TOKEN` (personal access token dengan akses baca repo).

Output (log, checkpoint, kurva ROC/PR, ringkasan metrik) tersimpan di `/kaggle/working/runs/`.

## 1. Ambil kode dari GitHub

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/akhyaruu/pediatric-kumadai.git"
BRANCH = "main"
REPO_DIR = "/tmp/pediatric-kumadai"  # outside /kaggle/working, so the code is not saved as notebook output

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # public repository or no secret attached
auth_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", auth_url, BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, auth_url, REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)  # no token on disk
subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=commit %h  %ci  %s"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

Cloning into '/tmp/pediatric-kumadai'...


commit 1deb059  2026-11-17 17:55:13 +0900  fix: kaggle setup


Updating files: 100% (39/39), done.


## 2. Paket yang belum ada di image Kaggle

In [2]:
import importlib.util

missing = [pip_name for module, pip_name in [("progressbar", "progressbar2")] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("missing packages installed:", missing or "none")

missing packages installed: none


## 3. Cek GPU, data, dan bobot ResNet-18

In [3]:
import kaggle_setup

kaggle_setup.print_environment()
DATA = kaggle_setup.find_data()
for key, path in DATA.items():
    print(f"{key:15s} {path}")
kaggle_setup.ensure_resnet_weights()

python 3.13.18 (/usr/bin/python3)
torch 2.11.0+cpu, CUDA build None
GPU: not available (Kaggle: Settings > Accelerator > GPU)


FileNotFoundError: folder 'preprocessed_400' not found under /kaggle/input
dicts_diagnosis/imputed/final/app_data_train not found under /kaggle/input
dicts_diagnosis/imputed/final/app_data_test not found under /kaggle/input
Add the preprocessed dataset to the notebook (Add Input) and check its folders.

## 4. Training + testing MVCBM

`SEEDS`: satu run per seed (paper melaporkan rata-rata beberapa seed). `OVERRIDES` dapat dipakai untuk mengubah isi config tanpa mengedit file, mis. `{"c_epochs": 2, "t_epochs": 2}` untuk uji cepat.

In [ ]:
import importlib

import train_mvcbm

importlib.reload(train_mvcbm)  # picks up a new version of the code after re-running step 1

CONFIG = "configs/mvcbm_seq_lstm_diag.yaml"
SEEDS = [42]
OVERRIDES = {}

results = {}
for seed in SEEDS:
    overrides = {**kaggle_setup.kaggle_overrides(DATA, run_name=f"mvcbm_seq_lstm_diag/seed{seed}"), **OVERRIDES}
    config, results[seed] = train_mvcbm.run(CONFIG, seed=seed, overrides=overrides)
    print(f"\nseed {seed}: outputs in {config['log_directory']}\n")

## 5. Ringkasan metrik test (epoch terakhir)

`c0` = Appendix_on_US, `c1` = Free_Fluids (urutan konsep di `prepare_data.py`).

In [ ]:
summary = kaggle_setup.summarize(results)
summary.to_csv("/kaggle/working/runs/mvcbm_seq_lstm_diag/summary.csv")
summary